In [ ]:
import cupy as cp  # 替换 NumPy 为 CuPy
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
import networkx as nx

def calculate_ccg(reference_spikes, target_spikes, time_window=[-0.02, 0.02], 
                 bin_size=0.001, jitter_window=0.005, n_jitters=100, 
                 confidence_level=0.99):
    """优化后的 CCG 计算，完全在 GPU 上运行"""
    # 将数据移到 GPU
    ref_spikes = cp.sort(cp.asarray(reference_spikes))
    targ_spikes = cp.sort(cp.asarray(target_spikes))
    bins = cp.arange(time_window[0], time_window[1] + bin_size, bin_size)
    
    # 原始 CCG：向量化时间差计算
    time_diffs = targ_spikes[None, :] - ref_spikes[:, None]  # 广播生成时间差矩阵
    mask = (time_diffs >= time_window[0]) & (time_diffs <= time_window[1])
    time_diffs = time_diffs[mask]
    original_ccg, _ = cp.histogram(time_diffs, bins=bins)
    
    # 抖动计算（在 GPU 上）
    jittered_ccgs = cp.empty((n_jitters, len(bins) - 1), dtype=cp.int32)
    for i in range(n_jitters):
        jitter = cp.random.uniform(-jitter_window, jitter_window, size=len(targ_spikes))
        jittered_spikes = targ_spikes + jitter
        diffs = jittered_spikes[None, :] - ref_spikes[:, None]
        mask = (diffs >= time_window[0]) & (diffs <= time_window[1])
        ccg, _ = cp.histogram(diffs[mask], bins=bins)
        jittered_ccgs[i] = ccg
    
    # 统计计算（在 GPU 上）
    jitter_mean = cp.mean(jittered_ccgs, axis=0)
    jitter_std = cp.std(jittered_ccgs, axis=0, ddof=1)
    global_upper = cp.quantile(jittered_ccgs, confidence_level)
    global_lower = cp.quantile(jittered_ccgs, 1 - confidence_level)
    
    # 转换回 NumPy 以进行后续操作
    original_ccg = cp.asnumpy(original_ccg)
    jitter_mean = cp.asnumpy(jitter_mean)
    jitter_std = cp.asnumpy(jitter_std)
    global_upper = cp.asnumpy(global_upper)
    global_lower = cp.asnumpy(global_lower)
    bin_centers = cp.asnumpy((bins[:-1] + bins[1:]) / 2)
    
    analysis_range = (bin_centers >= 0.001) & (bin_centers <= 0.005)
    analysis_ccg = original_ccg[analysis_range]
    
    analysis_range_pre = (bin_centers >= -0.02) & (bin_centers < 0)
    analysis_ccg_pre= original_ccg[analysis_range_pre]
    peak_height_pre = np.max(analysis_ccg_pre)
    
    peak_height = np.max(analysis_ccg)
    trough_depth = np.min(analysis_ccg)
    peak_idx = np.argmax(analysis_ccg)
    trough_idx = np.argmin(analysis_ccg)
    full_indices = np.where(analysis_range)[0]
    
    peak_full_idx = full_indices[peak_idx]
    trough_full_idx = full_indices[trough_idx]
    strength = (peak_height - jitter_mean[peak_full_idx]) / jitter_std[peak_full_idx] \
        if peak_height > trough_depth else \
        (trough_depth - jitter_mean[trough_full_idx]) / jitter_std[trough_full_idx]
    
    connection_type = "none"
    if (peak_height > global_upper*1.2) & (peak_height_pre < global_upper):
        connection_type = "excitatory"
    elif trough_depth < global_lower:
        connection_type = "inhibitory"
    
    return {
        'original_ccg': original_ccg,
        'bin_centers': bin_centers,
        'connection_type': connection_type,
        'connection_strength': strength,
        'jitter_mean': jitter_mean,
        'global_upper': global_upper,
        'global_lower': global_lower
    }

def pair_analysis(pair, spike_times):
    ref_idx, targ_idx = pair
    return pair, calculate_ccg(spike_times[ref_idx], spike_times[targ_idx])

def plot_ccg(pair, results, save_path=None, bin_size=0.001):
    ref_idx, targ_idx = pair
    plt.figure(figsize=(10, 6))
    plt.bar(results['bin_centers'] * 1000, results['original_ccg'], 
            width=bin_size * 1000, color='blue', alpha=0.7, label='CCG')
    plt.plot(results['bin_centers'] * 1000, results['jitter_mean'], 'r--', label='Jitter Mean')
    plt.axhline(y=results['global_upper'], color='g', linestyle='--', label='Upper Threshold')
    plt.axhline(y=results['global_lower'], color='g', linestyle='--', label='Lower Threshold')
    plt.xlabel('Time Lag (ms)')
    plt.ylabel('Spike Count')
    plt.title(f'CCG: Neuron {ref_idx} → Neuron {targ_idx}\n'
              f'Type: {results["connection_type"]}, Strength: {results["connection_strength"]:.2f}')
    plt.axvline(0, color='k', linestyle='--', alpha=0.3)
    plt.legend()
    plt.grid(True, alpha=0.3)
    if save_path:
        plt.savefig(f"{save_path}/ccg_pair_{ref_idx}_{targ_idx}.png")
        plt.close()
    else:
        plt.show()

def analyze_all_pairs_from_df(df, spike_column='spike_times', plot=False, save_path=None, batch_size=3):
    """分块处理神经元对，显示已完成对数的进度条，并跳过 mean_firing_rate > 15 的参考神经元以及 target 为 'pyramidal' 的情况。
    在内存不足时跳过当前批次并继续处理。
    """
    # Check if required columns exist
    if 'cell_type' not in df.columns:
        raise ValueError("DataFrame must contain 'cell_type' column")
    if 'mean_firing_rate' not in df.columns:
        raise ValueError("DataFrame must contain 'mean_firing_rate' column")
    spike_times = {idx: np.asarray(row[spike_column]) if isinstance(row[spike_column], (list, np.ndarray)) 
                   else np.array([row[spike_column]]) 
                   for idx, row in df.iterrows()}
    
    if len(spike_times) < 2:
        raise ValueError("Need at least 2 neurons for pair analysis")
    
    # Get mean_firing_rate and cell_type
    mean_firing_rates = df['mean_firing_rate']
    cell_types = df['cell_type']
    
    # Generate pairs, filtering out mean_firing_rate > 15 for reference and 'pyramidal' for target
    pairs = []
    # for ref_idx, targ_idx in combinations(df.index, 2):
    #     if ((mean_firing_rates[ref_idx] <= 15) and 
    #         (cell_types[targ_idx] != 'pyramidal')):
    #         pairs.append((ref_idx, targ_idx))

    for ref_idx, targ_idx in combinations(df.index, 2):
        if (mean_firing_rates[ref_idx] <= 15) & (cell_types[ref_idx]=='pyramidal') & (cell_types[targ_idx]!='pyramidal'):
            pairs.append((ref_idx, targ_idx))

    total_pairs = len(pairs)
    if total_pairs == 0:
        print("No pairs to process after filtering (all reference neurons have mean_firing_rate > 15 or all targets are pyramidal).")
        return {}
    
    results_dict = {}
    completed_pairs = 0
    
    # Process pairs in batches with progress bar
    for i in tqdm(range(0, len(pairs), batch_size), 
                  total=(len(pairs) + batch_size - 1) // batch_size,
                  desc="Processing pairs"):
        batch_pairs = pairs[i:i + batch_size]
        try:
            batch_results = [pair_analysis(pair, spike_times) for pair in batch_pairs]
            results_dict.update(dict(batch_results))
            completed_pairs += len(batch_pairs)
            tqdm.write(f"Completed {completed_pairs}/{total_pairs} pairs")
        except cp.cuda.memory.OutOfMemoryError as e:
            tqdm.write(f"OutOfMemoryError: Skipping batch {i//batch_size + 1} due to {str(e)}. Continuing with next batch...")
            # Free up GPU memory explicitly if possible
            cp.get_default_memory_pool().free_all_blocks()
            continue  # Skip this batch and move to the next
    
    # Batch plotting
    if plot and save_path:
        if not os.path.exists(save_path):
            os.makedirs(save_path)
        for pair in tqdm(results_dict, desc="Plotting", total=len(results_dict)):
            plot_ccg(pair, results_dict[pair], save_path)
    
    return results_dict

def update_connectivity(df, results, spike_column='spike_times'):
    # Initialize 'connectivity' and 'connection_pairs' columns

    df['connectivity'] = [None] * len(df)
    df['connection_pairs'] = [[] for _ in range(len(df))]  # Initialize as empty lists for each neuron
    
    # Iterate through the results to update connectivity and connection pairs
    for pair, result in results.items():
        if result['connection_type'] == 'excitatory':  # Only consider excitatory connections
            ref_idx, targ_idx = pair
            
            # Update 'connectivity' column
            if pd.isnull(df.iloc[ref_idx]['connectivity']):
                df.iloc[ref_idx, df.columns.get_loc('connectivity')] = 'pre'
            if pd.isnull(df.iloc[targ_idx]['connectivity']):
                df.iloc[targ_idx, df.columns.get_loc('connectivity')] = 'post'

            # Update 'connection_pairs' column: append pair info
            df.iloc[ref_idx, df.columns.get_loc('connection_pairs')].append(f"{ref_idx}->{targ_idx}")
            #df.iloc[targ_idx, df.columns.get_loc('connection_pairs')].append(f"{ref_idx}->{targ_idx}")
    
    # # Convert empty lists to None or keep as lists, depending on preference
    # df['connection_pairs'] = df['connection_pairs'].apply(lambda x: x if x else None)
    
    return df


import matplotlib.pyplot as plt
import numpy as np

def plot_ccg_of_pairs(df, save_path=None, bin_size=0.001, figsize=(10, 6), 
                     time_window=[-0.02, 0.02], jitter_window=0.005, n_jitters=10, 
                     confidence_level=0.99,spike_column="spike_times", pre_fix=None):
    """
    Plot CCGs for all excitatory connection pairs identified in the DataFrame, 
    recalculating CCGs from spike times.
    
    Parameters:
    - df: DataFrame with 'connection_pairs' and 'spike_times' columns
    - save_path: Optional directory to save the plots (e.g., 'path/to/save/')
    - bin_size: Bin size for CCG histogram
    - figsize: Tuple for figure size (width, height)
    - time_window, jitter_window, n_jitters, confidence_level: Parameters for calculate_ccg
    """
    # Ensure save_path exists if provided
    if save_path and not os.path.exists(save_path):
        os.makedirs(save_path)
    
    # Check required columns
    if 'connection_pairs' not in df.columns or 'spike_times' not in df.columns:
        raise ValueError("DataFrame must contain 'connection_pairs' and 'spike_times' columns")
    
    # Extract spike times
    spike_times = {idx: np.asarray(row[spike_column]) if isinstance(row[spike_column], (list, np.ndarray)) 
                   else np.array([row[spike_column]]) 
                   for idx, row in df.iterrows()}
    
    
    # Collect all unique pairs from 'connection_pairs'
    plotted_pairs = set()
    for idx, row in df.iterrows():
        for idx, row in df.iterrows():
            if (row['connection_pairs'] and 
                not isinstance(row['connection_pairs'], list) and 
                row['connection_pairs'] is not None):  # Check if there are any pairs
                try:
                    ref_idx, targ_idx = map(int, row['connection_pairs'].split('->'))
                    pair = (ref_idx, targ_idx)
                    print(pair)
                              
                    if pair not in plotted_pairs:

                        plotted_pairs.add(pair)
                        
                        # Recalculate CCG for this pair
                        result = calculate_ccg(spike_times[ref_idx], spike_times[targ_idx],
                                            time_window=time_window, bin_size=bin_size,
                                            jitter_window=jitter_window, n_jitters=n_jitters,
                                            confidence_level=confidence_level)
                        
                        # Plot the CCG
                        if result['connection_type'] is not None:
                            plt.figure(figsize=figsize)
                            plt.bar(result['bin_centers'] * 1000, result['original_ccg'], 
                                    width=bin_size * 1000, color='blue', alpha=0.7, label='CCG')
                            plt.plot(result['bin_centers'] * 1000, result['jitter_mean'], 
                                    'r--', label='Jitter Mean')
                            plt.axhline(y=result['global_upper'], color='g', linestyle='--', 
                                    label='Upper Threshold')
                            plt.axhline(y=result['global_lower'], color='g', linestyle='--', 
                                    label='Lower Threshold')
                            plt.xlabel('Time Lag (ms)')
                            plt.ylabel('Spike Count')
                            cell1 = df['cell_type'].iloc[ref_idx]
                            cell2 = df['cell_type'].iloc[targ_idx]
                            
                            plt.title(f'CCG: Neuron {ref_idx} → Neuron {targ_idx}\n'
                                    f'Type: {result["connection_type"]}, '
                                    f'Strength: {result["connection_strength"]:.2f}'
                                    f'cell type: ref:{cell1}, target:{cell2}')
                            plt.axvline(0, color='k', linestyle='--', alpha=0.3)
                            plt.legend()
                            plt.grid(True, alpha=0.3)
                            
                            # Save or show the plot
                            if save_path:
                                plt.savefig(f"{save_path}/{pre_fix}_{ref_idx}_{targ_idx}.png",
                                        bbox_inches='tight', dpi=300)
                                plt.close()
                            else:
                                plt.show()
                except ValueError:
                    print(f"Skipping invalid pair format: {row['connection_pairs']}")

                    continue
            elif isinstance(row['connection_pairs'], list) and row['connection_pairs']:
                    
                    for pair_str in row['connection_pairs']:
                        ref_idx, targ_idx = map(int, pair_str.split('->'))
                        pair = (ref_idx, targ_idx)
                        print(pair)
                              
                        if pair not in plotted_pairs:

                            plotted_pairs.add(pair)
                            
                            # Recalculate CCG for this pair
                            result = calculate_ccg(spike_times[ref_idx], spike_times[targ_idx],
                                                time_window=time_window, bin_size=bin_size,
                                                jitter_window=jitter_window, n_jitters=n_jitters,
                                                confidence_level=confidence_level)
                            
                            # Plot the CCG
                            if result['connection_type'] is not None:
                                plt.figure(figsize=figsize)
                                plt.bar(result['bin_centers'] * 1000, result['original_ccg'], 
                                        width=bin_size * 1000, color='blue', alpha=0.7, label='CCG')
                                plt.plot(result['bin_centers'] * 1000, result['jitter_mean'], 
                                        'r--', label='Jitter Mean')
                                plt.axhline(y=result['global_upper'], color='g', linestyle='--', 
                                        label='Upper Threshold')
                                plt.axhline(y=result['global_lower'], color='g', linestyle='--', 
                                        label='Lower Threshold')
                                plt.xlabel('Time Lag (ms)')
                                plt.ylabel('Spike Count')
                                plt.title(f'CCG: Neuron {ref_idx} → Neuron {targ_idx}\n'
                                        f'Type: {result["connection_type"]}, '
                                        f'Strength: {result["connection_strength"]:.2f}'
                                        f'cell type: ref:{cell1}, target:{cell2}')
                                plt.axvline(0, color='k', linestyle='--', alpha=0.3)
                                plt.legend()
                                plt.grid(True, alpha=0.3)
                                
                                # Save or show the plot
                                if save_path:
                                    plt.savefig(f"{save_path}/{pre_fix}_{ref_idx}_{targ_idx}.png",
                                            bbox_inches='tight', dpi=300)
                                    plt.close()
                                else:
                                    plt.show()


In [ ]:
def get_pkl_files(folder_path):
    # List all files in the directory
    all_files = os.listdir(folder_path)
    # Filter files that end with "withDLC.pkl"
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Example usage
#folder_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max"  # Replace with your actual folder path
adjust_path = r"S:\Sachuriga\file_with_table\ripple_ch"
out_path = r"S:\Sachuriga\file_with_table\ripple_ch/Functional_connections"
new_files = []
pkl_files = get_pkl_files(adjust_path)
erro_log=[]
for file in pkl_files:
#file=pkl_files[0]
    df_loaded = pd.read_pickle(fr'{adjust_path}/{file}').reset_index(drop=True)

    uniques = np.unique(df_loaded['session_id'])
    new_df = []
    for u in uniques:
        results = []
        temp = df_loaded[df_loaded['session_id']==u].copy()
        

        df_loaded['connectivity'] = None
        df_loaded['connection_pairs'] = None
        
        df_loaded[df_loaded['session_id']==u]['connectivity'] = [None] * len(df_loaded[df_loaded['session_id']==u])
        df_loaded[df_loaded['session_id']==u]['connection_pairs'] = [[] for _ in range(len(df_loaded[df_loaded['session_id']==u]))]  # Initialize as empty lists for each neuron

        try:
            results = analyze_all_pairs_from_df(temp, plot=False) 
                        # Iterate through the results to update connectivity and connection pairs
            for pair, result in results.items():
                if result['connection_type'] == 'excitatory':  # Only consider excitatory connections
                    ref_idx, targ_idx = pair
                    # Use .loc with the actual index values from the filtered DataFrame
                    df_loaded.iloc[ref_idx, df_loaded.columns.get_loc('connectivity')] = 'pre'
                    df_loaded.iloc[targ_idx, df_loaded.columns.get_loc('connectivity')] = 'post'

                    # Update 'connection_pairs' column: append pair info
                    df_loaded.iloc[ref_idx, df_loaded.columns.get_loc('connection_pairs')] = f"{ref_idx}->{targ_idx}"
        except Exception as e:
            erro_log.append((file,e))
    
    if os.path.exists(fr"{out_path}/{file}"):
        os.remove(fr"{out_path}/{file}")
    df_loaded.to_pickle(fr"{out_path}/{file}")


In [ ]:
file=r"63383_2024-07-10_15-37-51_units_table_withDLC.pkl"
df_loaded=pd.read_pickle(fr"S:\Sachuriga\file_with_table\ripple_ch\Functional_connections/{file}")
plot_ccg_of_pairs(df_loaded, save_path=fr'Q:\sachuriga\CR_CA1_paper\Results\connectivity', pre_fix = file.split("units_table_withDLC.pkl")[0])

In [ ]:
df_loaded

In [ ]:
import pandas as pd
def get_pkl_files(folder_path):
    # List all files in the directory
    all_files = os.listdir(folder_path)
    # Filter files that end with "withDLC.pkl"
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Example usage
#folder_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max"  # Replace with your actual folder path
adjust_path = r"S:\Sachuriga\file_with_table\ripple_ch\Functional_connections"
new_files = []
pkl_files = get_pkl_files(adjust_path)
erro_log=[]
for file in pkl_files:
    df_loaded = pd.read_pickle(fr"{adjust_path}/{file}")
    try:
        plot_ccg_of_pairs(df_loaded, save_path=fr'Q:\sachuriga\CR_CA1_paper\Results\connectivity',pre_fix = file.split("units_table_withDLC.pkl")[0])
    except Exception as e:
        continue

In [ ]:
import pandas as pd
def get_pkl_files(folder_path):
    # List all files in the directory
    all_files = os.listdir(folder_path)
    # Filter files that end with "withDLC.pkl"
    pkl_files = [f for f in all_files if f.endswith("withDLC.pkl")]
    return pkl_files

# Example usage
#folder_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max"  # Replace with your actual folder path
adjust_path = r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max\ripple_py"
new_files = []
pkl_files = get_pkl_files(adjust_path)
erro_log=[]
for file in pkl_files:
    df_loaded = pd.read_pickle(fr"{adjust_path}/{file}")

In [ ]:
import pandas as pd

temp = pd.read_pickle(r"S:\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel\ripple_max\ripple_py/63383_2024-07-10_A_units_table_withDLC.pkl")
temp

In [ ]:
df_loaded = pd.read_pickle(r'Q:/sachuriga/CR_CA1_paper/tables/functional_properties_with_python_measurements.pkl')
df_loaded 